# ARC-AGI-3 — Oracle calibration: our 47.39 bench is completion-bound, not efficiency-bound

**Status (stated honestly): offline analysis on the 25 public games — not a leaderboard score.**
We calibrated our production-configuration offline benchmark (25 public games x 4 passes = 100 runs,
mean **47.39**) against per-level *optimal* action counts, decomposing the missing points into
**completion loss vs efficiency loss** with the exact official score formula.

**Headline result:** of the 52.61 points we leave on the table, **50.30 are completion loss** and only
**2.30 are efficiency loss**. Even if every completed level were played in the provably minimum number
of actions, the bench would rise only to **49.70**. The bottleneck is levels we never finish — and
**25.6% of all our actions (5,432 of 21,217) are spent on levels that are never completed**.

Data sources: optimal routes from the public dataset
[maoxin123/arc3-25-optimal-solutions](https://www.kaggle.com/datasets/maoxin123/arc3-25-optimal-solutions)
(CC BY 4.0). Companion dataset published with this notebook: our step-by-step replay recordings of
those routes, [arc-agi-3-oracle-route-trajectories](https://www.kaggle.com/datasets/cosmorin/arc-agi-3-oracle-route-trajectories).
Our production kernel (harness lineage: Daniel Franzen's Duck harness + rellik13's Flash-Next/SGLang
serving recipe) is scheduled for submission in the 2026-10-27 UTC slot; until it is scored, our team's
leaderboard best remains 23.39. The leaderboard is computed on hidden games, so nothing here is a
leaderboard claim.


## The score formula (identified, then verified 100/100)

The production harness scores runs with `GameRun._compute_final_score` (a copy of the ARC-AGI
`EnvironmentScoreCalculator`, v0.9.8):

- **Level score** (completed, >= 1 action): `min(115, (baseline / actual_actions)^2 * 100)`; uncompleted levels score **0**.
- **Weights** are the level numbers (1-based). A game's score is the weighted mean of level scores,
  capped by `(sum of weights of scored levels) / (total weight) * 100`.
- `baseline` is the per-game human reference action count (`baseline_actions`).

We applied this formula to all 100 runs of our production benchmark and reproduced every recorded
`final_score` **with zero error (100/100)**. Every sensitivity number below is computed with this
exact formula.

Why this structure matters: once a level is finished *within* its baseline, extra inefficiency is
almost free — the 115 cap and the weight cap absorb it. Of our 134 completed levels, only **25** exceed
their baseline on average. The rest can be 2x, 3x, even 6x above the *optimal* route and still lose
**zero** points.


## Decomposition of the 47.39

| metric | value |
|---|---|
| Levels across 25 games | 183 |
| Completed in >= 1 of 4 passes | 134 |
| Never completed | 49 (concentrated: bp35: 8, lf52: 7, sk48: 7, su15: 5, wa30: 5) |
| Our mean actions on completed levels (total) | 5,488 |
| Optimal actions on the same levels (total) | 3,018 (ratio 1.82x) |
| Median per-level waste ratio (ours / optimal) | 1.67x |
| Levels where we beat the "optimal" route | **0** (consistency check: the routes are truly minimal) |
| Actions burned on never-completed levels | 5,432 of 21,217 = **25.6%** |
| **Efficiency loss** | **2.30 points** |
| **Completion loss** | **50.30 points** |

### Sensitivity: what efficiency fixes could buy (same completion set, official formula)

| scenario | bench mean | delta |
|---|---|---|
| S0 current | 47.39 | - |
| S1 halve the waste of the worst 10 levels | 48.60 | +1.21 |
| S2 eliminate the waste of the worst 10 levels | 49.38 | +1.98 |
| S3 halve the waste of all completed levels | 49.06 | +1.67 |
| S4 every completed level at optimal actions | 49.70 | **+2.30 (ceiling)** |

So the entire addressable efficiency upside is **+2.3 points**. Everything else is completion.


## Where the waste is: two repeatable patterns

Classified from the action streams of completed passes (movement-key repetition is genuine travel and
is *not* counted as waste; signals are repeated clicks on identical coordinates, RESET/UNDO use, and
direction-reversal adjacency). Worst 10 by bench gain if fixed:

| # | level | optimal | baseline | ours (mean) | ratio | completed | gain if waste removed |
|---|---|---|---|---|---|---|---|
| 1 | cd82 L5 | 13 | 23 | 36.0 | 2.77x | 4/4 | +0.43 |
| 2 | sc25 L3 | 11 | 32 | 57.8 | 5.25x | 4/4 | +0.35 |
| 3 | tu93 L8 | 21 | 23 | 51.3 | 2.44x | 3/4 | +0.34 |
| 4 | sc25 L4 | 21 | 83 | 100.5 | 4.79x | 4/4 | +0.32 |
| 5 | cd82 L2 | 6 | 8 | 19.5 | 3.25x | 4/4 | +0.29 |
| 6 | cd82 L4 | 13 | 21 | 33.0 | 2.54x | 4/4 | +0.27 |
| 7 | cd82 L6 | 16 | 23 | 23.0 | 1.44x | 4/4 | +0.25 |
| 8 | ka59 L3 | 33 | 51 | 69.8 | 2.11x | 4/4 | +0.18 |
| 9 | ka59 L4 | 38 | 51 | 57.0 | 1.50x | 3/4 | +0.14 |
| 10 | sc25 L1 | 13 | 36 | 77.5 | 5.96x | 4/4 | +0.10 |

**Pattern A — cyclic re-clicking of identical coordinates.** Spell-slot / palette cells are clicked
again and again in a fixed rotation: sc25 L3 repeats 85 of its 99 clicks on already-clicked
coordinates ((30,50)/(30,55)/(30,60) cycle), sc25 L1 repeats 169 of 207 clicks, and cd82 puts four
levels in the worst 10 from palette re-clicking alone. sc25 L4 adds a cell-by-cell grid sweep on top.

**Pattern B — RESET-separated identical retries.** tu93 L8 re-executes nearly the *same* movement
sequence three times, separated by RESETs (4 resets in the pass). The agent restarts the level but
not its plan.

Both patterns are detectable online ("state unchanged" / "same plan as last attempt"), which makes
them cheap guard/skill targets — worth doing, but remember the +2.3 ceiling.

### The frontier sink (never-completed levels eating the budget)

| level | actions spent (4 passes) | optimal | baseline |
|---|---|---|---|
| ka59 L6 | 385 | 45 | 132 |
| wa30 L5 | 363 | 94 | 368 |
| ls20 L2 | 355 | 45 | 123 |
| sc25 L5 | 318 | 36 | 143 |
| su15 L4 | 309 | **9** | 115 |
| re86 L7 | 305 | 98 | 424 |
| sk48 L1 | 218 | 14 | 61 |

su15 L4 is solvable in **9 actions**; we burned 309 across four passes and never finished it.
Cross-referencing agent traces with an oracle is what makes "structurally stuck on an easy level"
visible at all. Cutting losses earlier on such levels would recycle time budget into completions —
the only lever with real headroom.


## Leaderboard transfer estimate (revised, deliberately wider)

The bench is on public games, tuned while looking at public games; the LB runs on hidden games under
the submission budget. Because efficiency contributes only +/-2.3 points of variance, transfer is
almost entirely a question of **completion-rate transfer**:

| scenario | assumption | LB estimate |
|---|---|---|
| Optimistic | completion profile transfers as-is (bench ratio 0.95–1.0) | **45–50** |
| Neutral | completion rate drops 10–25% relative on unseen games (0.75–0.9) | **35–45** |
| Pessimistic | wall-game share rises, completion drops 30–45% relative (0.55–0.7) | **25–35** |

The gold line (~38.2 at the time of writing) sits *inside* the neutral band. In calibrated terms:
the bench margin over gold (47.39 - 38.2 = 9.2 points) is erased by a **~19% relative drop in
completion rate**. That is a more honest statement than "47.39 would be gold" — and only the actual
submission (2026-10-27 UTC slot) settles it.

## What we take from this

1. **Completion first.** The theoretical completion space is +50.3 points vs +2.3 for efficiency;
   effort allocation should reflect that ratio.
2. **Efficiency work, if any, is exactly two guards**: detect same-coordinate click cycles
   (Pattern A) and same-plan retries after RESET (Pattern B).
3. **Early give-up on frontier levels** converts sunk actions into budget for levels that can
   actually be finished — completion-side, not efficiency-side.
4. **Publish the oracle**: the replayed optimal-route trajectories (all 25 games, 5,502 steps,
   every game WIN at official-formula 100.00) are released as the companion dataset
   [arc-agi-3-oracle-route-trajectories](https://www.kaggle.com/datasets/cosmorin/arc-agi-3-oracle-route-trajectories)
   for anyone's self-distillation experiments.

## Credits

- **maoxin123** — the optimal route collection
  [arc3-25-optimal-solutions](https://www.kaggle.com/datasets/maoxin123/arc3-25-optimal-solutions)
  (CC BY 4.0) that this calibration and the companion trajectories are derived from.
- **Daniel Franzen** — the Duck agent harness our production stack builds on.
- **rellik13** — the verified Qwen Flash-Next + SGLang serving recipe our configuration follows.

Analysis by team cosmorin. Method note: waste classification is action-stream based; frame-diff
(no-op action rate) cross-checking is future work.
